In [1]:
import pandas as pd
from deep_translator import GoogleTranslator

from pathlib import Path
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import joblib

import pandas as pd
import re

START_YEAR = 2019

In [2]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

In [3]:
class Classifier(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, output_dim)
        )

    def forward(self, x):
        return self.net(x)

In [4]:
def embed(texts, batch_size=32):
    embeddings = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            encoded = tokenizer(batch, padding=True, truncation=True, return_tensors='pt').to(device)
            output = model(**encoded)
            cls_embeddings = output.last_hidden_state[:, 0, :]  # [CLS] token
            embeddings.append(cls_embeddings.cpu())
    return torch.cat(embeddings)

CIHR

In [5]:
cihr_path = "raw_data/CIHR/"
cihr_files = Path(cihr_path).glob("*.csv")

CIHR_DFS = [pd.read_csv(f) for f in cihr_files]
CIHR_DATA = pd.concat(CIHR_DFS, ignore_index=True)

In [6]:
grant_descriptors = [
    "ApplicationTitle_TitreDemande", "PrimaryThemeEN_ThemePrincipalAN", "AllResearchCategoriesEN_TousCategoriesRechercheAN", "ApplicationKeywords_MotsClesDemande"
]

col_names = [
    'Title', 'Main_Discipline', 'Area_of_Research', 'Keywords'
]

CIHR_DATA = CIHR_DATA[grant_descriptors]
CIHR_DATA.columns = col_names

CIHR_DATA.drop_duplicates(inplace=True)
CIHR_DATA["Main_Discipline"].value_counts()

Main_Discipline
Biomedical                                         8989
Clinical                                           3869
Social/Cultural/Environmental/Population Health    3003
Health systems/services                            2830
Not applicable/Specified                            127
Name: count, dtype: int64

Training CIHR Main Discipline

In [7]:
tmp_data = CIHR_DATA.sample(frac=1).reset_index(drop=True) # shuffle

# tmp_data = tmp_data.groupby(["Main_Discipline"]).head(500)
tmp_data = tmp_data[tmp_data["Main_Discipline"] != "Not applicable/Specified"]

tmp_data = tmp_data.dropna(subset=['Main_Discipline'])
tmp_data["Main_Discipline"].value_counts()


Main_Discipline
Biomedical                                         8989
Clinical                                           3869
Social/Cultural/Environmental/Population Health    3003
Health systems/services                            2830
Name: count, dtype: int64

In [8]:
label_mapping = dict(enumerate(tmp_data['Main_Discipline'].astype('category').cat.categories))
tmp_data['Main_Discipline'] = tmp_data['Main_Discipline'].astype('category').cat.codes

X_train_text, X_test_text, y_train, y_test = train_test_split(
    tmp_data['Title'], tmp_data['Main_Discipline'], test_size=0.2, stratify=tmp_data['Main_Discipline'], #random_state=42
)

X_train = embed(X_train_text.astype(str).tolist())
X_test = embed(X_test_text.astype(str).tolist())
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

input_dim = X_train.shape[1]
output_dim = len(label_mapping)

clf_model = Classifier(input_dim, output_dim).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(clf_model.parameters(), lr=1e-4)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [9]:
# Train
for epoch in range(500):
    clf_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = clf_model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 251.6965
Epoch 2: Loss = 182.7230
Epoch 3: Loss = 165.7223
Epoch 4: Loss = 159.8401
Epoch 5: Loss = 156.3835
Epoch 6: Loss = 153.6858
Epoch 7: Loss = 152.5726
Epoch 8: Loss = 150.5121
Epoch 9: Loss = 149.6844
Epoch 10: Loss = 147.8990
Epoch 11: Loss = 146.6178
Epoch 12: Loss = 146.1460
Epoch 13: Loss = 144.8293
Epoch 14: Loss = 144.1445
Epoch 15: Loss = 143.1770
Epoch 16: Loss = 142.6667
Epoch 17: Loss = 141.1442
Epoch 18: Loss = 140.7617
Epoch 19: Loss = 139.8058
Epoch 20: Loss = 138.9485
Epoch 21: Loss = 138.1325
Epoch 22: Loss = 137.7944
Epoch 23: Loss = 136.6298
Epoch 24: Loss = 136.0376
Epoch 25: Loss = 135.3613
Epoch 26: Loss = 134.4633
Epoch 27: Loss = 133.7903
Epoch 28: Loss = 133.1778
Epoch 29: Loss = 132.5248
Epoch 30: Loss = 132.0213
Epoch 31: Loss = 130.9518
Epoch 32: Loss = 130.4912
Epoch 33: Loss = 129.5192
Epoch 34: Loss = 128.3610
Epoch 35: Loss = 128.0598
Epoch 36: Loss = 127.4158
Epoch 37: Loss = 126.9521
Epoch 38: Loss = 125.7401
Epoch 39: Loss = 125.

In [10]:
clf_model.eval()
with torch.no_grad():
    preds = clf_model(X_test.to(device))
    pred_labels = preds.argmax(dim=1).cpu().numpy()
    true_labels = y_test.numpy()

print(classification_report(true_labels, pred_labels, target_names=list(label_mapping.values()), zero_division=0))

                                                 precision    recall  f1-score   support

                                     Biomedical       0.88      0.90      0.89      1798
                                       Clinical       0.60      0.59      0.59       774
                        Health systems/services       0.63      0.61      0.62       566
Social/Cultural/Environmental/Population Health       0.69      0.66      0.68       601

                                       accuracy                           0.75      3739
                                      macro avg       0.70      0.69      0.69      3739
                                   weighted avg       0.75      0.75      0.75      3739



In [11]:
torch.save(clf_model.state_dict(), "models/CIHR_MD.pt")
joblib.dump(label_mapping, "models/CIHR_MD_label_mapping.pkl")


['models/CIHR_MD_label_mapping.pkl']

NSERC

In [12]:
nserc_path = "raw_data/NSERC/"
nserc_files = Path(nserc_path).glob("*.csv")

NSERC_DFS = [pd.read_csv(f) for f in nserc_files]
NSERC_DATA = pd.concat(NSERC_DFS, ignore_index=True)

In [13]:
grant_descriptors = [
    "ApplicationTitle", "AreaOfApplicationGroupEN", "ResearchSubjectEN", "Keyword"
]

col_names = [
     'Title', 'Main_Discipline', 'Area_of_Research', 'Keywords'
]

NSERC_DATA = NSERC_DATA[grant_descriptors]
NSERC_DATA.columns = col_names

NSERC_DATA.drop_duplicates(inplace=True)

Model for NSERC Main Discipline

In [14]:
tmp_data = NSERC_DATA.sample(frac=1).reset_index(drop=True) # shuffle

tmp_data = tmp_data[tmp_data["Main_Discipline"] != "Not available"]

tmp_data["Main_Discipline"].value_counts()

Main_Discipline
Advancement of knowledge                   20177
Manufacturing processes and products        4626
Information and communication services      3936
Environment                                 3927
Energy resources                            3123
Health, education and social services       2937
Transportation systems and services         2077
Agriculture and primary food production     1541
Construction, urban and rural planning      1376
Northern development                        1203
Natural resources (economic aspects)        1106
The socioeconomic objective available        737
Commercial services                          400
Name: count, dtype: int64

In [15]:
label_mapping = dict(enumerate(tmp_data['Main_Discipline'].astype('category').cat.categories))
tmp_data['Main_Discipline'] = tmp_data['Main_Discipline'].astype('category').cat.codes

X_train_text, X_test_text, y_train, y_test = train_test_split(
    tmp_data['Title'], tmp_data['Main_Discipline'], test_size=0.2, stratify=tmp_data['Main_Discipline'], #random_state=42
)

X_train = embed(X_train_text.tolist())
X_test = embed(X_test_text.tolist())
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

input_dim = X_train.shape[1]
output_dim = len(label_mapping)

clf_model = Classifier(input_dim, output_dim).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(clf_model.parameters(), lr=1e-4)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [16]:
# Train
for epoch in range(500):
    clf_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = clf_model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 1044.2470
Epoch 2: Loss = 807.6514
Epoch 3: Loss = 764.5913
Epoch 4: Loss = 745.0504
Epoch 5: Loss = 732.1668
Epoch 6: Loss = 722.0283
Epoch 7: Loss = 714.9733
Epoch 8: Loss = 706.5693
Epoch 9: Loss = 701.1718
Epoch 10: Loss = 694.0983
Epoch 11: Loss = 688.9203
Epoch 12: Loss = 683.7209
Epoch 13: Loss = 679.3596
Epoch 14: Loss = 673.9651
Epoch 15: Loss = 669.1330
Epoch 16: Loss = 665.1938
Epoch 17: Loss = 660.9550
Epoch 18: Loss = 654.9392
Epoch 19: Loss = 651.5486
Epoch 20: Loss = 648.0804
Epoch 21: Loss = 644.5676
Epoch 22: Loss = 640.2824
Epoch 23: Loss = 635.8041
Epoch 24: Loss = 632.4406
Epoch 25: Loss = 627.9209
Epoch 26: Loss = 624.2464
Epoch 27: Loss = 621.7219
Epoch 28: Loss = 618.2678
Epoch 29: Loss = 614.5122
Epoch 30: Loss = 610.2624
Epoch 31: Loss = 606.3806
Epoch 32: Loss = 603.3069
Epoch 33: Loss = 599.8961
Epoch 34: Loss = 596.0714
Epoch 35: Loss = 593.7689
Epoch 36: Loss = 588.6428
Epoch 37: Loss = 586.2057
Epoch 38: Loss = 584.0101
Epoch 39: Loss = 581

In [17]:
clf_model.eval()
with torch.no_grad():
    preds = clf_model(X_test.to(device))
    pred_labels = preds.argmax(dim=1).cpu().numpy()
    true_labels = y_test.numpy()

print(classification_report(true_labels, pred_labels, target_names=list(label_mapping.values()), zero_division=0))

                                         precision    recall  f1-score   support

               Advancement of knowledge       0.82      0.89      0.85      4036
Agriculture and primary food production       0.80      0.75      0.77       308
                    Commercial services       0.65      0.55      0.59        80
 Construction, urban and rural planning       0.76      0.72      0.74       275
                       Energy resources       0.82      0.80      0.81       625
                            Environment       0.77      0.76      0.76       786
  Health, education and social services       0.81      0.68      0.74       588
 Information and communication services       0.84      0.81      0.82       787
   Manufacturing processes and products       0.69      0.71      0.70       925
   Natural resources (economic aspects)       0.79      0.63      0.70       221
                   Northern development       0.79      0.61      0.69       241
  The socioeconomic objecti

In [18]:
torch.save(clf_model.state_dict(), "models/NSERC_MD.pt")
joblib.dump(label_mapping, "models/NSERC_MD_label_mapping.pkl")

['models/NSERC_MD_label_mapping.pkl']

Model for NSERC Area of Research

In [19]:
tmp_data = NSERC_DATA.sample(frac=1).reset_index(drop=True) # shuffle
translator = GoogleTranslator(target="en")
                              
val_counts = tmp_data["Area_of_Research"].value_counts()
classes = val_counts[val_counts > 50].index
tmp_data = tmp_data[tmp_data["Area_of_Research"].isin(classes)]

for class_name in classes:
    tmp_data["Area_of_Research"] = tmp_data["Area_of_Research"].replace(class_name, translator.translate(class_name))

val_counts = tmp_data["Area_of_Research"].value_counts()
classes = val_counts[val_counts > 100].index
tmp_data = tmp_data[tmp_data["Area_of_Research"].isin(classes)]

tmp_data['Area_of_Research'] = tmp_data['Area_of_Research'].str.replace(r' \(.*?\)', '', regex=True)
                              
tmp_data = tmp_data.groupby(["Area_of_Research"]).head(500)
tmp_data = tmp_data[tmp_data["Area_of_Research"] != "Not available"]

tmp_data = tmp_data.dropna(subset=['Area_of_Research'])
tmp_data["Area_of_Research"].value_counts()

Area_of_Research
Animal ecology                    500
Psychology                        500
Biochemistry                      500
Civil engineering                 500
Artificial intelligence           500
                                 ... 
Animal nutrition and husbandry    105
Photonics                         104
Enzymes                           102
Fuel and energy technology        102
Database management               101
Name: count, Length: 135, dtype: int64

In [20]:
label_mapping = dict(enumerate(tmp_data['Area_of_Research'].astype('category').cat.categories))
tmp_data['Area_of_Research'] = tmp_data['Area_of_Research'].astype('category').cat.codes

X_train_text, X_test_text, y_train, y_test = train_test_split(
    tmp_data['Title'], tmp_data['Area_of_Research'], test_size=0.2, stratify=tmp_data['Area_of_Research'], #random_state=42
)

X_train = embed(X_train_text.astype(str).tolist())
X_test = embed(X_test_text.astype(str).tolist())
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

input_dim = X_train.shape[1]
output_dim = len(label_mapping)

clf_model = Classifier(input_dim, output_dim).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(clf_model.parameters(), lr=1e-4)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [21]:
# Train
for epoch in range(500):
    clf_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = clf_model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 2216.6477
Epoch 2: Loss = 1877.0745
Epoch 3: Loss = 1629.1016
Epoch 4: Loss = 1502.3559
Epoch 5: Loss = 1429.9016
Epoch 6: Loss = 1380.9239
Epoch 7: Loss = 1350.3946
Epoch 8: Loss = 1324.6930
Epoch 9: Loss = 1305.6159
Epoch 10: Loss = 1287.9462
Epoch 11: Loss = 1275.1708
Epoch 12: Loss = 1260.9282
Epoch 13: Loss = 1248.6673
Epoch 14: Loss = 1239.4330
Epoch 15: Loss = 1231.6160
Epoch 16: Loss = 1224.1065
Epoch 17: Loss = 1216.0125
Epoch 18: Loss = 1208.4842
Epoch 19: Loss = 1203.9054
Epoch 20: Loss = 1196.4796
Epoch 21: Loss = 1187.4747
Epoch 22: Loss = 1183.5732
Epoch 23: Loss = 1177.2204
Epoch 24: Loss = 1173.3451
Epoch 25: Loss = 1166.1567
Epoch 26: Loss = 1160.2837
Epoch 27: Loss = 1158.5731
Epoch 28: Loss = 1152.3328
Epoch 29: Loss = 1150.1332
Epoch 30: Loss = 1147.3543
Epoch 31: Loss = 1144.2563
Epoch 32: Loss = 1135.1522
Epoch 33: Loss = 1133.2645
Epoch 34: Loss = 1129.7412
Epoch 35: Loss = 1124.1632
Epoch 36: Loss = 1122.7094
Epoch 37: Loss = 1117.3077
Epoch 38: 

In [22]:
clf_model.eval()
with torch.no_grad():
    preds = clf_model(X_test.to(device))
    pred_labels = preds.argmax(dim=1).cpu().numpy()
    true_labels = y_test.numpy()
    
print(classification_report(true_labels, pred_labels, target_names=list(label_mapping.values()), zero_division=0))

                                                            precision    recall  f1-score   support

                                    Advanced manufacturing       0.24      0.22      0.23        32
        Aerospace, aeronautical and automotive engineering       0.37      0.34      0.35        92
                                  Agricultural engineering       0.17      0.18      0.18        22
                                                Algorithms       0.32      0.21      0.25        34
                                      Analytical chemistry       0.31      0.47      0.37       100
                                            Animal biology       0.12      0.11      0.12       100
                                            Animal ecology       0.40      0.51      0.45       100
                            Animal nutrition and husbandry       0.50      0.33      0.40        21
                          Animal physiology and metabolism       0.27      0.24      0.25       100

In [23]:
torch.save(clf_model.state_dict(), "models/NSERC_AR.pt")
joblib.dump(label_mapping, "models/NSERC_AR_label_mapping.pkl")

['models/NSERC_AR_label_mapping.pkl']

SSHRC

In [24]:
sshrc_path = "raw_data/SSHRC/"
sshrc_files = Path(sshrc_path).glob("*.csv")

SSHRC_DFS = [pd.read_csv(f) for f in sshrc_files]
SSHRC_DATA = pd.concat(SSHRC_DFS, ignore_index=True)

In [25]:
grant_descriptors = [
    "Title-Titre", "Area_of_Research", "Main_Discipline", "Keywords-Mots-clés"
]

col_names = [
    'Title', 'Main_Discipline', 'Area_of_Research', 'Keywords'
]


SSHRC_DATA = SSHRC_DATA[grant_descriptors]
SSHRC_DATA.columns = col_names

SSHRC_DATA.drop_duplicates(inplace=True)

Model for SSHRC Main Discipline

In [26]:
tmp_data = SSHRC_DATA.sample(frac=1).reset_index(drop=True) # shuffle

# tmp_data = tmp_data.groupby(["Main_Discipline"]).head(500)
tmp_data = tmp_data[~(tmp_data['Main_Discipline'].isin(["Not Specified", "Not Subject to Research Classification"]))]

val_counts = tmp_data["Main_Discipline"].value_counts()
classes = val_counts[val_counts > 100].index
tmp_data = tmp_data[tmp_data["Main_Discipline"].isin(classes)]

tmp_data = tmp_data.dropna(subset=['Main_Discipline'])
tmp_data = tmp_data.groupby(["Main_Discipline"]).head(500)
tmp_data["Main_Discipline"].value_counts()

Main_Discipline
Post-Secondary Education and Research                   500
Indigenous peoples                                      500
Information Technologies                                500
Women                                                   500
Youth                                                   500
Gender Issues                                           500
Science and technology                                  500
Law and Justice                                         500
Environment and Sustainability                          500
Communication                                           500
Education                                               500
Economic and Regional Development                       500
Employment and labour                                   500
Family                                                  500
Violence                                                500
Health                                                  500
Management              

In [27]:
label_mapping = dict(enumerate(tmp_data['Main_Discipline'].astype('category').cat.categories))
tmp_data['Main_Discipline'] = tmp_data['Main_Discipline'].astype('category').cat.codes

X_train_text, X_test_text, y_train, y_test = train_test_split(
    tmp_data['Title'], tmp_data['Main_Discipline'], test_size=0.2, stratify=tmp_data['Main_Discipline'], #random_state=42
)

X_train = embed(X_train_text.astype(str).tolist())
X_test = embed(X_test_text.astype(str).tolist())
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

input_dim = X_train.shape[1]
output_dim = len(label_mapping)

clf_model = Classifier(input_dim, output_dim).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(clf_model.parameters(), lr=1e-4)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [28]:
# Train
for epoch in range(500):
    clf_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = clf_model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 808.7523
Epoch 2: Loss = 742.0342
Epoch 3: Loss = 665.7343
Epoch 4: Loss = 607.7191
Epoch 5: Loss = 568.5125
Epoch 6: Loss = 544.7133
Epoch 7: Loss = 526.3057
Epoch 8: Loss = 513.6452
Epoch 9: Loss = 504.2871
Epoch 10: Loss = 498.5094
Epoch 11: Loss = 491.4332
Epoch 12: Loss = 485.7350
Epoch 13: Loss = 480.2449
Epoch 14: Loss = 476.7856
Epoch 15: Loss = 472.3546
Epoch 16: Loss = 468.9070
Epoch 17: Loss = 467.0415
Epoch 18: Loss = 462.8325
Epoch 19: Loss = 459.8496
Epoch 20: Loss = 456.7925
Epoch 21: Loss = 455.2665
Epoch 22: Loss = 453.8178
Epoch 23: Loss = 450.5665
Epoch 24: Loss = 448.1152
Epoch 25: Loss = 446.5514
Epoch 26: Loss = 444.9138
Epoch 27: Loss = 442.5403
Epoch 28: Loss = 440.1531
Epoch 29: Loss = 439.1773
Epoch 30: Loss = 436.5577
Epoch 31: Loss = 435.3623
Epoch 32: Loss = 433.2690
Epoch 33: Loss = 431.2172
Epoch 34: Loss = 430.1117
Epoch 35: Loss = 428.5810
Epoch 36: Loss = 427.2769
Epoch 37: Loss = 424.9550
Epoch 38: Loss = 424.8375
Epoch 39: Loss = 421.

In [29]:
clf_model.eval()
with torch.no_grad():
    preds = clf_model(X_test.to(device))
    pred_labels = preds.argmax(dim=1).cpu().numpy()
    true_labels = y_test.numpy()
    
print(classification_report(true_labels, pred_labels, target_names=list(label_mapping.values()), zero_division=0))

                                                      precision    recall  f1-score   support

                                         Agriculture       0.66      0.78      0.72        50
                                    Arts and culture       0.36      0.38      0.37       100
                         Canada's Official Languages       0.50      0.52      0.51        21
                                            Children       0.53      0.56      0.54       100
                                  Children and youth       0.08      0.03      0.05        30
                                       Communication       0.45      0.44      0.45       100
                   Economic and Regional Development       0.49      0.49      0.49       100
                                           Education       0.44      0.44      0.44       100
                                             Elderly       0.77      0.87      0.82        93
                               Employment and labour       

In [30]:
torch.save(clf_model.state_dict(), "models/SSHRC_MD.pt")
joblib.dump(label_mapping, "models/SSHRC_MD_label_mapping.pkl")

['models/SSHRC_MD_label_mapping.pkl']

Model for SSHRC Area of Research

In [31]:
SSHRC_DATA["Area_of_Research"].value_counts()

tmp_data = SSHRC_DATA.sample(frac=1).reset_index(drop=True) # shuffle

val_counts = tmp_data["Area_of_Research"].value_counts()
classes = val_counts[val_counts > 200].index
tmp_data = tmp_data[tmp_data["Area_of_Research"].isin(classes)]

tmp_data = tmp_data.groupby(["Area_of_Research"]).head(500)
tmp_data = tmp_data.dropna(subset=['Area_of_Research'])
tmp_data = tmp_data[~(tmp_data['Area_of_Research'].isin(["Not Specified", "Not specified", "Not Applicable", "Multiple primary fields of research", "Interdisciplinary Studies"]))]


tmp_data["Area_of_Research"].value_counts()


Area_of_Research
Education                                            500
Fine Arts                                            500
Geography                                            500
Economics                                            500
Philosophy                                           500
Anthropology                                         500
Management, Business, Administrative Studies         500
Archaeology                                          500
Urban and Regional Studies, Environmental Studies    500
Communications and Media Studies                     500
Social Work                                          500
Criminology                                          500
Literature, Modern Languages and                     500
Law                                                  500
Sociology                                            500
Political Science                                    500
History                                              500
Psychology    

In [32]:
label_mapping = dict(enumerate(tmp_data['Area_of_Research'].astype('category').cat.categories))
tmp_data['Area_of_Research'] = tmp_data['Area_of_Research'].astype('category').cat.codes

X_train_text, X_test_text, y_train, y_test = train_test_split(
    tmp_data['Title'], tmp_data['Area_of_Research'], test_size=0.2, stratify=tmp_data['Area_of_Research'], #random_state=42
)

X_train = embed(X_train_text.astype(str).tolist())
X_test = embed(X_test_text.astype(str).tolist())
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

input_dim = X_train.shape[1]
output_dim = len(label_mapping)

clf_model = Classifier(input_dim, output_dim).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(clf_model.parameters(), lr=1e-4)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [33]:
# Train
for epoch in range(500):
    clf_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = clf_model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 413.6638
Epoch 2: Loss = 393.0314
Epoch 3: Loss = 363.1119
Epoch 4: Loss = 332.0365
Epoch 5: Loss = 306.1774
Epoch 6: Loss = 288.5539
Epoch 7: Loss = 275.2565
Epoch 8: Loss = 266.3786
Epoch 9: Loss = 258.9398
Epoch 10: Loss = 252.9876
Epoch 11: Loss = 248.3107
Epoch 12: Loss = 244.3996
Epoch 13: Loss = 241.0960
Epoch 14: Loss = 237.1773
Epoch 15: Loss = 234.5058
Epoch 16: Loss = 232.1830
Epoch 17: Loss = 229.7364
Epoch 18: Loss = 227.1366
Epoch 19: Loss = 225.1462
Epoch 20: Loss = 223.5747
Epoch 21: Loss = 222.1519
Epoch 22: Loss = 220.0450
Epoch 23: Loss = 219.1166
Epoch 24: Loss = 217.4241
Epoch 25: Loss = 216.2019
Epoch 26: Loss = 214.6994
Epoch 27: Loss = 213.4053
Epoch 28: Loss = 212.4640
Epoch 29: Loss = 211.6612
Epoch 30: Loss = 209.4994
Epoch 31: Loss = 208.9673
Epoch 32: Loss = 208.7909
Epoch 33: Loss = 207.2882
Epoch 34: Loss = 206.6645
Epoch 35: Loss = 205.5237
Epoch 36: Loss = 203.9405
Epoch 37: Loss = 203.3797
Epoch 38: Loss = 201.5591
Epoch 39: Loss = 201.

In [34]:
clf_model.eval()
with torch.no_grad():
    preds = clf_model(X_test.to(device))
    pred_labels = preds.argmax(dim=1).cpu().numpy()
    true_labels = y_test.numpy()
    
print(classification_report(true_labels, pred_labels, target_names=list(label_mapping.values()), zero_division=0))

                                                   precision    recall  f1-score   support

                                     Anthropology       0.32      0.35      0.33       100
                                      Archaeology       0.71      0.77      0.74       100
             Classics, Classical & Dead Languages       0.66      0.57      0.61        54
                 Communications and Media Studies       0.44      0.43      0.44       100
                                      Criminology       0.62      0.74      0.68       100
                                       Demography       0.39      0.37      0.38        52
                                        Economics       0.65      0.62      0.63       100
                                        Education       0.60      0.53      0.56       100
                                        Fine Arts       0.41      0.40      0.41       100
                                        Geography       0.34      0.36      0.35       10

In [35]:
torch.save(clf_model.state_dict(), "models/SSHRC_AR.pt")
joblib.dump(label_mapping, "models/SSHRC_AR_label_mapping.pkl")

['models/SSHRC_AR_label_mapping.pkl']